# BF2 — Graph Analytics

Builds and analyzes NetworkX graphs from the museum KG:
co-visitation, flow, user-artwork bipartite.
Centrality analysis, community detection, influential artworks.

In [ ]:
import os
import sys
import networkx as nx
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('helpers'))
from graph_builder import TrajectoNXGraphBuilder, build_all_graphs, print_graph_summary

## 1. Build All Graph Types

In [ ]:
builder = TrajectoNXGraphBuilder()
graphs = build_all_graphs()
print_graph_summary(graphs)

## 2. Co-visitation Graph Analysis

In [ ]:
G_covisit = graphs['poi_covisit']
if G_covisit.number_of_nodes() > 0:
    # Degree centrality
    degree_cent = nx.degree_centrality(G_covisit)
    top_artworks = sorted(degree_cent.items(), key=lambda x: x[1], reverse=True)[:10]
    print('Top 10 artworks by co-visitation degree centrality:')
    for uri, cent in top_artworks:
        short = uri.split('#')[-1] if '#' in uri else uri.split('/')[-1]
        print(f'  {short}: {cent:.4f} (degree={G_covisit.degree(uri)})')

    # Betweenness centrality
    betw_cent = nx.betweenness_centrality(G_covisit)
    top_betw = sorted(betw_cent.items(), key=lambda x: x[1], reverse=True)[:10]
    print('\nTop 10 by betweenness centrality:')
    for uri, cent in top_betw:
        short = uri.split('#')[-1] if '#' in uri else uri.split('/')[-1]
        print(f'  {short}: {cent:.4f}')

## 3. Community Detection

In [ ]:
if G_covisit.number_of_nodes() > 0:
    from networkx.algorithms.community import greedy_modularity_communities

    communities = list(greedy_modularity_communities(G_covisit))
    print(f'Communities found: {len(communities)}')
    for i, comm in enumerate(communities[:5]):
        print(f'  Community {i+1}: {len(comm)} artworks')
        sample = list(comm)[:3]
        for uri in sample:
            short = uri.split('#')[-1] if '#' in uri else uri.split('/')[-1]
            print(f'    - {short}')

## 4. POI Flow Graph

In [ ]:
G_flow = graphs['poi_flow']
if G_flow.number_of_nodes() > 0:
    # PageRank
    pr = nx.pagerank(G_flow)
    top_pr = sorted(pr.items(), key=lambda x: x[1], reverse=True)[:10]
    print('Top 10 artworks by PageRank (flow graph):')
    for uri, score in top_pr:
        short = uri.split('#')[-1] if '#' in uri else uri.split('/')[-1]
        print(f'  {short}: {score:.4f}')

    # Most common transitions
    edges = sorted(G_flow.edges(data=True), key=lambda x: x[2].get('weight', 0), reverse=True)
    print('\nTop 10 artwork transitions:')
    for u, v, d in edges[:10]:
        u_short = u.split('#')[-1] if '#' in u else u.split('/')[-1]
        v_short = v.split('#')[-1] if '#' in v else v.split('/')[-1]
        print(f'  {u_short} -> {v_short}: {d.get("weight", 0)} times')

## 5. User Similarity Graph

In [ ]:
G_user_sim = graphs['user_similarity']
if G_user_sim.number_of_nodes() > 0:
    stats = builder.get_graph_statistics(G_user_sim)
    print('User similarity graph:')
    for k, v in stats.items():
        print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

    # Most connected users
    degree_seq = sorted(G_user_sim.degree(), key=lambda x: x[1], reverse=True)
    print('\nMost connected visitors:')
    for uri, deg in degree_seq[:10]:
        short = uri.split('#')[-1] if '#' in uri else uri.split('/')[-1]
        print(f'  {short}: {deg} shared-artwork connections')

## 6. Degree Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, G) in zip(axes, [('Co-visitation', G_covisit),
                                  ('User Similarity', G_user_sim),
                                  ('User-POI', graphs['user_poi'])]):
    if G.number_of_nodes() > 0:
        degrees = [d for _, d in G.degree()]
        ax.hist(degrees, bins=20, edgecolor='black', alpha=0.7)
        ax.set_title(f'{name} Degree Distribution')
        ax.set_xlabel('Degree')
        ax.set_ylabel('Count')

plt.tight_layout()
plt.show()